In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:

from utils.funciones_minio import crear_cliente_minio, bajar_minio
from utils.config import MINIO_EMBEDDINGS   
EMBEDDINGS_IMAGENES = "embeddings_imagenes.parquet"

In [3]:
client = crear_cliente_minio()

In [10]:
embedding_imagenes = bajar_minio(client, MINIO_EMBEDDINGS, EMBEDDINGS_IMAGENES)

In [11]:
x = np.stack(embedding_imagenes["embedding"].values)
y = embedding_imagenes["clase"].values.to_numpy()
print(x.shape)
print(y.shape)

(172306, 2048)
(172306,)


In [1]:
# 1. Convertir etiquetas de texto a números
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# 2. Convertir a formato categórico (One-Hot)
y_categorical = tf.keras.utils.to_categorical(y_encoded, num_classes=4)

# 3. Split: 80% Train, 20% Temporal (que dividiremos en Val y Test)
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)

# 4. Split del temporal: 50% Val, 50% Test (esto da 10% y 10% del total)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Estructura de entrenamiento: {x_train.shape}")
print(f"Clases detectadas: {encoder.classes_}")

NameError: name 'LabelEncoder' is not defined